# Projet 5 — Phases 1 et 2 : problématique, données, protocole et baseline

**Sujet** : détection d'attaques inconnues par Anomaly Detection sur UNSW-NB15.

**Question centrale** : *peut-on détecter des attaques inconnues sans disposer d'exemples de ces attaques pendant l'entraînement ?*

Ce notebook couvre les deux premières phases du projet :

| Phase | Durée indicative | Contenu du sujet | Sections du notebook |
|---|---|---|---|
| **Phase 1** | 2 h | Problématique, compréhension du dataset, EDA ciblée | 1.1 à 1.5 |
| **Phase 2** | 2 h | Préparation des données, pipeline et baseline | 2.1 à 2.6 |

Les expériences 1 à 3 (phases 3 à 5) s'appuieront directement sur ce qui est construit ici : le découpage des données, la règle de seuil, les métriques, le pipeline de prétraitement et la baseline.

> Les cellules marquées ✍️ sont à compléter avec **vos** chiffres après exécution.

# PHASE 1 — Problématique, compréhension du dataset, EDA ciblée

## 1.1 Problématique et hypothèses

Un SOC (centre opérationnel de sécurité) veut une méthode de détection qui **n'exige pas d'exemples de toutes les attaques possibles**. L'idée testée : apprendre uniquement le **comportement normal** du trafic, puis signaler ce qui s'en écarte.

**Hypothèses à tester dans les phases suivantes** (formulées dès maintenant, avant toute expérience) :

| | Hypothèse | Testée en |
|---|---|---|
| **H1** | Les méthodes de détection d'anomalies auront des compromis différents entre détection et faux positifs | Expérience 1 (phase 3) |
| **H2** | Plus le jeu d'entraînement « normal » est contaminé par des attaques, moins les modèles distinguent normal et attaque | Expérience 2 (phase 4) |
| **H3** | Certaines méthodes d'anomalies seront moins précises globalement qu'un classifieur supervisé, mais plus utiles sur certaines familles inconnues | Expérience 3 (phase 5) |

**Pourquoi formuler les hypothèses avant les expériences ?** Une hypothèse écrite après avoir vu les résultats s'adapte toujours à ce qu'on a observé : elle ne teste plus rien. En la fixant maintenant, on s'oblige à la confirmer ou à l'infirmer honnêtement.

## Pourquoi cette approche ?

### Pourquoi la détection d'anomalies ?
Un classifieur supervisé apprend à reconnaître les attaques **qu'on lui a montrées**. Il ne peut pas, par construction, apprendre une attaque qui n'existait pas au moment de l'entraînement. La détection d'anomalies inverse le problème : on modélise ce qui est **normal**, et tout ce qui s'en écarte devient suspect. C'est la seule famille de méthodes qui peut, en principe, répondre à la question centrale. Le prix à payer est connu : tout ce qui est inhabituel n'est pas malveillant, d'où un risque de fausses alertes, que l'on mesure tout au long du notebook.

### Pourquoi UNSW-NB15 ?
- Il est plus récent et plus réaliste que KDD Cup 99 / NSL-KDD, longtemps utilisés mais jugés obsolètes.
- Il fournit la **famille** de chaque attaque (`attack_cat`, 9 familles). C'est indispensable pour l'expérience 3, qui retire des familles entières de l'apprentissage.
- Il propose un **train et un test officiels séparés** par ses auteurs : on évite de mesurer la performance sur des flux presque identiques à ceux de l'entraînement.
- Limite assumée : le trafic est **généré en laboratoire** (outil IXIA PerfectStorm), avec des artefacts connus (TTL, à étudier dans les phases suivantes).

## Registre des choix méthodologiques (phases 1 et 2)

Ce tableau rassemble les décisions prises dans ce notebook, leur justification et l'alternative écartée. Il sert de support pour défendre la méthode en soutenance.

| Choix | Valeur retenue | Justification | Alternative écartée (et pourquoi) | Section |
|---|---|---|---|---|
| Train / test | Fichiers officiels, le plus gros = train | Respecte le découpage des auteurs ; le test est réellement séparé de l'entraînement | Découper un seul fichier au hasard : flux du test presque identiques à ceux du train → scores trop optimistes | 1.3 |
| Validation | 20 % du train officiel, stratifiée par famille | ~35 000 flux dont ~11 000 normaux : un quantile à 1 % est estimé sur ~110 flux, donc stable. La stratification garde les familles rares dans les mêmes proportions | 10 % : seuils plus instables. Validation croisée : coût ×5 avec One-Class SVM, pour un gain limité | 2.2 |
| Doublons | Retirés **seulement** de l'entraînement des détecteurs | LOF calcule des distances entre voisins : des flux identiques donnent des distances nulles et des densités infinies. Validation et test gardent les doublons pour refléter le vrai trafic | Dédoublonner partout : fausserait les taux mesurés (un flux répété 100 fois compte 100 fois en réalité) | 2.2 |
| Taille d'entraînement | `N_FIT` = 20 000 flux normaux, identique pour tous | One-Class SVM a un coût d'entraînement entre quadratique et cubique en nombre de points, LOF fait une recherche de voisins par flux scoré. 20 000 flux décrivent bien le normal et les 60 entraînements de l'expérience 2 restent faisables | Tout le pool normal (~45 000 flux) : One-Class SVM trop lent. Taille différente par modèle : comparaison inéquitable | 1.2, 2.2 |
| Prétraitement | Appris sur l'entraînement de chaque modèle (`Pipeline`) | Exigence anti-fuite du sujet. En détection d'anomalies, standardiser sur le normal revient à **définir** ce qu'est un flux normal | Standardiser sur tout le dataset : le test influencerait les transformations (fuite) | 2.3 |
| Valeurs extrêmes | Écrêtage au 99e percentile | Protège les moyennes, écarts-types et distances de quelques valeurs démesurées, tout en gardant 1 % de valeurs extrêmes, qui sont souvent **le signal** recherché | 95e percentile (notebook d'origine) : écrase 5 % des valeurs, donc efface des anomalies. Aucun écrêtage : quelques valeurs énormes dominent l'échelle | 2.3 |
| Asymétrie | `log1p` sur les variables continues | Octets, paquets, durées s'étalent sur plusieurs ordres de grandeur ; le log les ramène à une échelle comparable. `log1p` accepte la valeur 0 | Box-Cox : refuse les zéros. `QuantileTransformer` : uniformise tout et efface les écarts entre valeurs extrêmes | 2.3 |
| Mise à l'échelle | `StandardScaler` sur toutes les variables numériques | LOF, One-Class SVM et la baseline reposent sur des distances : sans mise à l'échelle, la variable aux plus grandes valeurs décide seule | `RobustScaler` : possible, mais l'écrêtage et le log ont déjà neutralisé les extrêmes | 2.3 |
| Catégorielles | One-hot des 5 modalités les plus fréquentes + « rare » | Le top 5 couvre la grande majorité des flux ; peu de colonnes en plus, ce qui protège les méthodes à distance du fléau de la dimension. Une modalité jamais vue va dans « rare » | One-hot complet (> 130 colonnes pour `proto`, presque toutes vides). Encodage ordinal : crée un ordre artificiel entre protocoles, qui fausse les distances | 2.3 |
| Baseline | Écart moyen en z-score au trafic normal | Le détecteur le plus simple possible : si les modèles avancés ne font pas mieux, ils n'apportent rien | Baseline « aucune alerte » : triviale, rappel nul, n'apprend rien | 2.4 |
| Seuil d'alerte | Quantile `1 − TARGET_FPR` des scores des **normaux de validation** | N'utilise aucune attaque ; met tous les modèles au **même budget de fausses alertes** ; correspond au raisonnement d'un SOC | Paramètre `contamination` d'IF ou seuil 0 d'OC-SVM : non comparables entre modèles. Seuil maximisant le F1 avec les labels : utilise des attaques, donc optimiste | 2.1 |
| FPR visé | 5 %, complété par 1, 2 et 10 % | Produit assez d'alertes pour mesurer un rappel fiable ; les autres niveaux montrent le compromis détection / fausses alertes | 1 % seul : rappels très faibles, comparaisons bruitées | 1.2, 2.5 |

## 1.2 Configuration et reproductibilité

Tous les paramètres sont regroupés ici pour être visibles et modifiables à un seul endroit.

| Paramètre | Valeur | Rôle et justification |
|---|---|---|
| `RANDOM_STATE` | 42 | Graine commune (découpages, échantillonnages) : relancer le notebook donne les mêmes résultats. La valeur elle-même n'a pas d'importance, seul le fait de la fixer compte |
| `VAL_SIZE` | 0,2 | Assez de flux normaux (~11 000) pour estimer un seuil à 1 % de FPR sur une centaine de flux |
| `N_FIT` | 20 000 | Taille du jeu d'entraînement « normal ». Elle sera identique pour tous les modèles des phases suivantes (équité), et limitée par le coût de calcul de LOF et One-Class SVM |
| `TARGET_FPR` | 5 % | Budget de fausses alertes servant à fixer le seuil (section 2.1) |
| `FPR_LEVELS` | 1, 2, 5, 10 % | Budgets testés pour décrire le compromis détection / fausses alertes |
| `UNKNOWN_FAMILIES` | 4 familles | Familles prévues comme « inconnues » pour l'expérience 3 ; on vérifie dès l'EDA qu'elles ont assez de flux de test |
| `CLIP_QUANTILE` | 0,99 | Écrêtage des valeurs extrêmes (section 2.3) |
| `FAST_MODE` | désactivé | Version rapide pour vérifier que le notebook tourne ; résultats non exploitables pour le rapport |

In [ ]:
import os
import time
import warnings
import platform

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import sklearn

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, PrecisionRecallDisplay, RocCurveDisplay,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

# ---------------- Paramètres (justifiés dans le tableau ci-dessus) ----------------
RANDOM_STATE = 42
DATA_DIR = os.environ.get("UNSW_DATA_DIR", "/kaggle/input/unsw-nb15")
TRAIN_FILE = "UNSW_NB15_training-set.csv"
TEST_FILE = "UNSW_NB15_testing-set.csv"
OUT_DIR = "results"                    # tableaux et figures pour le rapport

FAST_MODE = os.environ.get("UNSW_FAST", "0") == "1"

VAL_SIZE = 0.2
N_FIT = 2_000 if FAST_MODE else 20_000
TARGET_FPR = 0.05
FPR_LEVELS = [0.01, 0.02, 0.05, 0.10]
UNKNOWN_FAMILIES = ["Reconnaissance", "Backdoor", "Shellcode", "Worms"]
CLIP_QUANTILE = 0.99

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(RANDOM_STATE)

def save_fig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), bbox_inches="tight")

def save_table(df, name):
    df.to_csv(os.path.join(OUT_DIR, f"{name}.csv"))

# ---------------- Versions (à reporter dans le README / rapport) ----------------
versions = {
    "Python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__, "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__, "seaborn": sns.__version__,
}
print("Mode rapide :", FAST_MODE)
pd.Series(versions, name="version").to_frame()

## 1.3 Chargement et compréhension du dataset

**Dataset** : UNSW-NB15, version partitionnée (UNSW Canberra, 2015). Chaque ligne est un **flux réseau** décrit par 42 variables, avec `label` (0 normal / 1 attaque) et `attack_cat` (famille d'attaque).

Sur Kaggle, les deux fichiers sont souvent **inversés** (le fichier *training* contient 82 332 lignes). On prend le plus gros comme train officiel (175 341 lignes), ce qui correspond au découpage des auteurs.

## 1.4 EDA ciblée

L'EDA est **ciblée** : on ne décrit pas tout le dataset, on répond seulement aux questions qui conditionnent le protocole de la phase 2 :
1. Combien de flux normaux pour apprendre le comportement normal ?
2. Quelles familles d'attaques existent et en quelle quantité dans le test (assez d'exemples pour l'expérience 3) ?
3. Y a-t-il des doublons qui fausseraient un apprentissage par voisinage (LOF) ?
4. Les attaques se distinguent-elles visiblement du normal sur quelques variables clés ?

In [ ]:
df_a = pd.read_csv(os.path.join(DATA_DIR, TRAIN_FILE))
df_b = pd.read_csv(os.path.join(DATA_DIR, TEST_FILE))
if len(df_a) >= len(df_b):
    train_df, test_df = df_a.copy(), df_b.copy()
else:
    train_df, test_df = df_b.copy(), df_a.copy()
    print("Fichiers inversés détectés : le fichier 'testing' sert de train officiel.")

for df in (train_df, test_df):
    df["attack_cat"] = df["attack_cat"].astype(str).str.strip()

FEATURES = [c for c in train_df.columns if c not in ("id", "attack_cat", "label")]
CAT_COLS = ["proto", "service", "state"]

print(f"Train officiel : {len(train_df):>7} flux | Test officiel : {len(test_df):>7} flux | {len(FEATURES)} variables")
print("Valeurs manquantes :", int(train_df.isna().sum().sum()), "/", int(test_df.isna().sum().sum()))

**Variables du dataset, regroupées par nature** (utile pour comprendre ce que « normal » veut dire ici) :

| Groupe | Exemples | Ce qu'elles décrivent |
|---|---|---|
| Identification du flux | `proto`, `service`, `state` | Protocole (tcp, udp…), service applicatif (http, dns…), état de la connexion (FIN, CON, INT…) |
| Volumes | `spkts`, `dpkts`, `sbytes`, `dbytes` | Paquets et octets envoyés par la source (`s`) et la destination (`d`) |
| Durée et débit | `dur`, `rate`, `sload`, `dload` | Durée du flux, paquets par seconde, débits en bits par seconde |
| Temporelles | `sinpkt`, `dinpkt`, `sjit`, `djit`, `tcprtt`, `synack`, `ackdat` | Temps entre paquets, gigue, temps d'établissement de la connexion TCP |
| Paquets et TCP | `sttl`, `dttl`, `swin`, `dwin`, `smean`, `dmean`, `sloss`, `dloss` | TTL, taille de fenêtre TCP, taille moyenne des paquets, pertes |
| Applicatives | `trans_depth`, `response_body_len`, `ct_flw_http_mthd`, `is_ftp_login`, `ct_ftp_cmd` | Activité HTTP et FTP |
| Contexte (`ct_*`) | `ct_srv_src`, `ct_dst_ltm`, `ct_src_dport_ltm`… | Nombre de connexions similaires parmi les 100 dernières (même service, même destination…) |
| Cibles | `label`, `attack_cat` | 0 normal / 1 attaque, et famille de l'attaque |

**Point important pour la suite** : `label` et `attack_cat` ne sont **jamais** donnés aux détecteurs comme variables d'entrée. `attack_cat` sert uniquement à construire les expériences (familles inconnues) et à analyser les résultats par famille.

In [ ]:
fam = pd.DataFrame({
    "train": train_df["attack_cat"].value_counts(),
    "test": test_df["attack_cat"].value_counts(),
}).fillna(0).astype(int).sort_values("train", ascending=False)
fam["inconnue (Exp. 3)"] = fam.index.isin(UNKNOWN_FAMILIES)

dup_normal = train_df[train_df["label"] == 0].duplicated(subset=FEATURES).mean()
print(f"Part de flux normaux dupliqués dans le train : {dup_normal:.1%}")

fig, ax = plt.subplots(figsize=(8, 4.5))
fam[["train", "test"]].plot(kind="barh", logx=True, ax=ax)
ax.invert_yaxis()
ax.set_title("Flux par famille (échelle log)")
ax.set_xlabel("Nombre de flux")
plt.tight_layout(); save_fig(fig, "eda_familles"); plt.show()
fam

**Lecture du code ci-dessus**
- `value_counts()` compte les flux par famille, dans le train et dans le test ; `fillna(0)` met 0 si une famille manque dans un fichier.
- La colonne `inconnue (Exp. 3)` signale les familles prévues comme inconnues : on vérifie dès maintenant qu'elles ont assez de flux de **test** pour que leur taux de détection ait un sens.
- `duplicated(subset=FEATURES).mean()` donne la part de flux normaux identiques à un flux déjà vu, sur les 42 variables (sans `id`, toujours différent).
- L'échelle logarithmique est indispensable : les familles vont de plusieurs dizaines de milliers de flux à quelques dizaines.

In [ ]:
KEY_VARS = [c for c in ["dur", "sbytes", "dbytes", "rate", "sttl", "smean"] if c in FEATURES]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.ravel(), KEY_VARS):
    sns.boxplot(data=train_df.assign(classe=train_df["label"].map({0: "Normal", 1: "Attaque"})),
                x="classe", y=np.log1p(train_df[col]), ax=ax, showfliers=False)
    ax.set_title(f"log(1 + {col})"); ax.set_xlabel(""); ax.set_ylabel("")
plt.suptitle("Normal vs attaque sur quelques variables clés (train officiel)")
plt.tight_layout(); save_fig(fig, "eda_variables_cles"); plt.show()

**Pourquoi ces graphiques ?** Ils vérifient l'hypothèse de base de toute l'approche : si les attaques avaient exactement les mêmes distributions que le trafic normal, aucun détecteur d'anomalies ne pourrait les repérer. Les boîtes montrent, variable par variable, si les attaques se situent ailleurs que le normal. On passe en `log(1 + x)` car les valeurs s'étendent sur plusieurs ordres de grandeur.

Attention : un écart visible sur une variable seule ne prouve pas qu'elle **cause** la détection, et deux distributions qui se chevauchent sur chaque variable peuvent quand même se séparer quand on combine les variables.

## 1.5 Conclusions de la phase 1 : ce que l'EDA implique pour le protocole

- Le trafic normal est abondant : on peut apprendre le comportement normal sur un échantillon de `N_FIT` flux.
- Les familles rares (Worms, Shellcode…) ont peu d'exemples dans le test : leurs taux de détection auront une **forte incertitude** (à signaler dans les limites).
- Les doublons normaux sont nombreux : on les retire du jeu d'entraînement des détecteurs, sinon LOF calcule des distances nulles entre flux identiques et ses scores deviennent instables. Les doublons restent dans la validation et le test pour refléter le trafic réel.
- Les distributions sont très asymétriques (d'où le log dans le prétraitement), et certaines variables comme `sttl` séparent fortement les classes : ce point sera testé dans l'analyse approfondie (phase 6).

✍️ *À compléter : 2–3 observations chiffrées tirées de vos graphiques (proportion d'attaques, familles les plus rares dans le test, variables les plus discriminantes visuellement).*

# PHASE 2 — Préparation des données, pipeline et baseline

## 2.1 Protocole expérimental

### Découpage des données

| Jeu | Contenu | Usage |
|---|---|---|
| **Pool d'entraînement normal** | Flux **normaux** de 80 % du train officiel, sans doublons | Entraînement de la baseline puis des détecteurs (échantillon de `N_FIT` flux) |
| **Pool d'attaques d'entraînement** | Attaques des mêmes 80 % | Mis de côté : servira à **contaminer** l'entraînement (Exp. 2) et à entraîner le classifieur supervisé (Exp. 3) |
| **Validation** | 20 % restants du train officiel (stratifiés par famille) | Fixer les seuils (sur ses flux normaux), choisir le modèle à analyser |
| **Test** | Fichier test officiel | **Uniquement** pour mesurer les résultats finaux |

**Utilisation du test.** Aucune décision (seuil, hyperparamètre, choix du modèle) n'est prise sur le test. Dans ce notebook, le test sert uniquement à donner un **point de référence** pour la baseline (section 2.5) ; aucun choix n'en dépend.

### Règle de seuil (commune à tous les modèles)

Un détecteur d'anomalies produit un **score** : plus il est élevé, plus le flux est jugé anormal. Il faut un seuil pour transformer ce score en alerte.

On fixe ce seuil **sans utiliser d'attaques** : c'est le quantile `1 − TARGET_FPR` des scores des flux **normaux de validation**. Avec `TARGET_FPR = 5 %`, chaque modèle laisse passer 95 % du trafic normal de validation sans alerte.

Pourquoi ce choix :
- c'est cohérent avec la question posée (on ne dispose pas d'exemples d'attaques pour régler le système) ;
- c'est réaliste pour un SOC, qui raisonne en **volume d'alertes acceptable** ;
- tous les modèles sont comparés **au même budget de faux positifs**, donc équitablement.

### Métriques

L'attaque est la classe positive. L'accuracy n'est **jamais** utilisée seule, car elle est trompeuse avec des classes déséquilibrées.

| Métrique | Question à laquelle elle répond |
|---|---|
| **Rappel** | Quelle part des attaques est détectée ? |
| **Précision** | Parmi les alertes, quelle part sont de vraies attaques ? |
| **F1** et **Macro-F1** | Compromis précision/rappel ; Macro-F1 donne le même poids aux classes normale et attaque |
| **FPR** | Quelle part du trafic normal déclenche une fausse alerte ? |
| **PR-AUC** | Qualité du classement des scores, indépendamment du seuil (adaptée au déséquilibre) |
| **ROC-AUC** | Capacité globale à séparer normal et attaque, indépendamment du seuil |

### Conditions comparables entre modèles

Mêmes variables, même prétraitement, **même échantillon d'entraînement**, même règle de seuil, mêmes graines. Les hyperparamètres sont fixés **a priori** (valeurs usuelles de la littérature, justifiées lors des expériences) et ne sont pas optimisés sur le test.

### Pourquoi ce protocole ?

**Pourquoi trois jeux (entraînement, validation, test) et pas deux ?**
Chaque fois qu'on règle quelque chose en regardant des résultats (un seuil, un choix de modèle), ces résultats deviennent optimistes. Il faut donc un jeu pour **régler** (validation) et un autre, jamais regardé avant la fin, pour **mesurer** (test). Si l'on réglait le seuil sur le test, le score final mesurerait en partie notre ajustement plutôt que la vraie qualité du modèle.

**Pourquoi la validation vient du train officiel et pas du test ?**
Le test officiel doit rester intact. Prendre la validation dans le train garantit qu'aucune information du test n'intervient dans les décisions.

**Pourquoi stratifier la validation par famille (`attack_cat`) et pas seulement par `label` ?**
Une stratification par `label` garde la proportion normal/attaque, mais pourrait laisser presque aucun *Worms* dans la validation. En stratifiant par famille, chaque famille est représentée dans la même proportion qu'au départ.

**Pourquoi entraîner sur un échantillon de flux normaux et non sur tout le train ?**
- Le scénario étudié suppose qu'on ne dispose **pas** d'exemples d'attaques : les détecteurs ne voient que du normal.
- `N_FIT` est identique pour tous les modèles, condition d'une comparaison équitable (exigée par le sujet).

**Pourquoi le seuil est-il fixé uniquement avec des flux normaux ?**
C'est la seule règle cohérente avec la question posée. Si l'on utilisait des attaques étiquetées pour choisir le seuil, on supposerait connaître les attaques, ce que l'approche cherche justement à éviter. Cette règle a aussi un sens opérationnel : un SOC fixe d'abord **combien de fausses alertes il peut traiter**, puis mesure ce que ce budget permet de détecter.

### Rappel des formules

Avec TP = attaques détectées, FN = attaques manquées, FP = flux normaux signalés à tort, TN = flux normaux non signalés :

| Métrique | Formule | Lecture |
|---|---|---|
| Rappel (taux de détection) | TP / (TP + FN) | 80 % = 8 attaques sur 10 détectées |
| Précision | TP / (TP + FP) | 60 % = 4 alertes sur 10 sont fausses |
| F1 | 2 × Précision × Rappel / (Précision + Rappel) | Élevé seulement si les deux sont élevés |
| Macro-F1 | Moyenne du F1 de la classe normale et du F1 de la classe attaque | Pénalise un modèle bon sur une seule classe |
| FPR | FP / (FP + TN) | 1 % = 1 flux normal sur 100 déclenche une alerte |
| PR-AUC | Aire sous la courbe précision-rappel, pour tous les seuils possibles | Vaut la proportion d'attaques pour un modèle aléatoire, 1 pour un modèle parfait |
| ROC-AUC | Probabilité qu'une attaque tirée au hasard ait un score plus élevé qu'un flux normal tiré au hasard | 0,5 = hasard, 1 = parfait |

**Pourquoi PR-AUC plutôt que ROC-AUC comme critère principal ?** La ROC-AUC intègre le FPR, dont le dénominateur est le grand nombre de flux normaux : quelques milliers de fausses alertes y paraissent négligeables. La PR-AUC mesure directement la qualité des alertes (précision), ce qui compte pour un analyste.

## 2.2 Découpage des données

Mise en œuvre du découpage décrit ci-dessus. Points à remarquer dans le code :
- `stratify=train_df["attack_cat"]` : la validation garde la même proportion de chaque famille que le train officiel ;
- `drop_duplicates(subset=FEATURES)` n'est appliqué qu'au **pool d'entraînement normal**, pas à la validation ni au test ;
- `X_fit` est tiré **une seule fois** avec la graine fixe : c'est ce même échantillon qui servira à la baseline et aux trois détecteurs de l'expérience 1 ;
- les variables numériques sont réparties en **continues** (plus de 50 valeurs distinctes) et **discrètes**, ce qui détermine leur traitement en 2.3.

In [ ]:
train_part, val_df = train_test_split(
    train_df, test_size=VAL_SIZE, stratify=train_df["attack_cat"], random_state=RANDOM_STATE)

normal_pool = train_part[train_part["label"] == 0].drop_duplicates(subset=FEATURES)
attack_pool = train_part[train_part["label"] == 1]

X_val, y_val = val_df[FEATURES], val_df["label"].to_numpy()
X_test, y_test = test_df[FEATURES], test_df["label"].to_numpy()
val_normal = y_val == 0
X_val_normal = X_val[val_normal]

N_FIT = min(N_FIT, len(normal_pool))
X_fit = normal_pool.sample(n=N_FIT, random_state=RANDOM_STATE)[FEATURES]

num_cols = [c for c in FEATURES if c not in CAT_COLS]
CONT_COLS = [c for c in num_cols if normal_pool[c].nunique() > 50]
DISC_COLS = [c for c in num_cols if c not in CONT_COLS]

splits = pd.DataFrame({
    "flux": [len(normal_pool), len(attack_pool), N_FIT, len(val_df), len(test_df)],
    "dont attaques": [0, len(attack_pool), 0, int(y_val.sum()), int(y_test.sum())],
}, index=["Pool normal (sans doublons)", "Pool attaques", "Entraînement détecteurs (N_FIT)",
          "Validation", "Test officiel"])
save_table(splits, "decoupage")
splits

**Contrôles de non-fuite.** La cellule suivante vérifie automatiquement que le protocole est respecté : aucun flux d'attaque dans l'entraînement de la baseline, aucun recouvrement entre entraînement et validation, et un test qui vient bien d'un autre fichier.

In [ ]:
assert (train_df.loc[X_fit.index, "label"] == 0).all(), "Une attaque s'est glissée dans l'entraînement"
assert X_fit.index.isin(train_part.index).all(), "L'entraînement doit venir des 80 % du train officiel"
assert not X_fit.index.isin(val_df.index).any(), "Recouvrement entre entraînement et validation"
assert not X_fit.duplicated().any(), "Il reste des doublons dans l'entraînement"
print("Contrôles réussis : entraînement 100 % normal, sans doublon, disjoint de la validation.")
print(f"Validation : {int((y_val == 0).sum())} flux normaux, {int(y_val.sum())} attaques")
print(f"Test officiel (fichier séparé) : {int((y_test == 0).sum())} flux normaux, {int(y_test.sum())} attaques")

## 2.3 Pipeline de prétraitement

Le prétraitement est intégré dans un `Pipeline` scikit-learn avec chaque modèle. Toutes ses statistiques (percentiles, moyennes, écarts-types, catégories gardées) sont donc apprises **sur le seul jeu d'entraînement** du modèle, puis appliquées telles quelles à la validation et au test. C'est ce qui évite toute fuite de données (pas de scaling ni de sélection sur l'ensemble complet).

| Variables | Transformation | Raison |
|---|---|---|
| Continues (> 50 valeurs distinctes) | Écrêtage au 99e percentile → `log1p` → standardisation | Distributions à très longue queue ; les détecteurs à base de distance (LOF, OC-SVM) y sont très sensibles |
| Discrètes (compteurs `ct_*`, TTL, indicateurs) | Standardisation | Même échelle pour toutes les variables |
| Catégorielles (`proto`, `service`, `state`) | One-hot des 5 modalités les plus fréquentes + « rare » | Évite l'explosion du nombre de colonnes ; une modalité jamais vue va dans « rare » |

Pour la baseline et les détecteurs, le prétraitement est appris **uniquement sur du trafic normal** : une valeur ou un protocole inhabituel pour le trafic normal ressort alors naturellement comme anormal.

#### Pourquoi ces transformations, dans cet ordre ?

1. **Écrêtage d'abord** : on plafonne les valeurs démesurées avant tout calcul, pour qu'elles ne faussent ni le log ni la moyenne et l'écart-type du `StandardScaler`. Le seuil (99e percentile) est **appris sur l'entraînement** : un flux de test plus grand que tout ce qui a été vu est ramené à cette valeur maximale, il reste donc parmi les plus extrêmes.
2. **Log ensuite** : sur des variables comme `sbytes` (de quelques dizaines d'octets à plusieurs millions), une différence de 100 octets n'a pas le même sens à 200 qu'à 2 000 000. Le log compare des **rapports** plutôt que des différences, ce qui correspond mieux à la façon dont ces grandeurs varient.
3. **Standardisation enfin** : chaque variable a alors une moyenne de 0 et un écart-type de 1 **sur le trafic normal**. Une valeur à +3 signifie « 3 écarts-types au-dessus du normal », ce qui donne un sens direct aux distances calculées par LOF, One-Class SVM et la baseline.

**Pourquoi distinguer variables continues et discrètes ?** Les compteurs (`ct_*`), indicateurs (`is_*`) et TTL prennent peu de valeurs entières : les écrêter ou leur appliquer un log n'apporte rien et peut écraser des valeurs significatives (par exemple un TTL de 254). Le seuil de 50 valeurs distinctes, repris du notebook d'origine, sert seulement à séparer ces deux types de variables.

**Pourquoi le même prétraitement pour Isolation Forest, qui n'en a pas besoin ?** Isolation Forest est insensible aux transformations monotones (log, mise à l'échelle) car il coupe sur des seuils. Garder le même prétraitement pour tous assure que seules les **méthodes** diffèrent entre les modèles, pas les données qu'elles reçoivent.

**Pourquoi un `Pipeline` ?** Il garantit mécaniquement que `fit` n'est appelé que sur les données d'entraînement, et que validation et test passent par exactement les mêmes transformations. C'est plus sûr qu'une série d'étapes manuelles, où une erreur de variable suffit à créer une fuite.

In [ ]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Plafonne chaque colonne à un quantile appris sur les données d'entraînement."""
    def __init__(self, upper_quantile=0.99):
        self.upper_quantile = upper_quantile
    def fit(self, X, y=None):
        self.upper_ = np.quantile(np.asarray(X, dtype=float), self.upper_quantile, axis=0)
        return self
    def transform(self, X):
        return np.minimum(np.asarray(X, dtype=float), self.upper_)
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def log1p_pos(X):
    return np.log1p(np.clip(X, 0, None))


def build_preprocessor(cont_cols, disc_cols, cat_cols):
    cont_pipe = Pipeline([
        ("clip", QuantileClipper(CLIP_QUANTILE)),
        ("log", FunctionTransformer(log1p_pos, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])
    return ColumnTransformer([
        ("cont", cont_pipe, cont_cols),
        ("disc", StandardScaler(), disc_cols),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", max_categories=6), cat_cols),
    ], sparse_threshold=0)


def make_pipeline(model, cols=None):
    cont, disc, cat = cols or (CONT_COLS, DISC_COLS, CAT_COLS)
    return Pipeline([("prep", build_preprocessor(cont, disc, cat)), ("model", model)])

### Vérifications du pipeline

Deux contrôles pour montrer que le pipeline fait ce qu'on attend :

1. **Effet sur une variable** : `sbytes` avant et après écrêtage + log.
2. **Preuve que tout est appris sur l'entraînement seulement** : après transformation, les variables continues ont une moyenne de 0 et un écart-type de 1 **sur `X_fit`** (c'est là que le `StandardScaler` a été ajusté). Sur la validation et le test, elles ne valent pas exactement 0 et 1 : les statistiques n'ont pas été recalculées sur ces jeux. Si elles l'étaient, ce serait une fuite de données.

In [ ]:
col = "sbytes"
clip = QuantileClipper(CLIP_QUANTILE).fit(X_fit[[col]])
after = log1p_pos(clip.transform(X_fit[[col]])).ravel()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(X_fit[col], bins=60)
axes[0].set_title(f"{col} brut (flux normaux d'entraînement)")
axes[1].hist(after, bins=60)
axes[1].set_title(f"{col} après écrêtage {CLIP_QUANTILE:.0%} + log1p")
plt.tight_layout(); save_fig(fig, "phase2_effet_pretraitement"); plt.show()

prep = build_preprocessor(CONT_COLS, DISC_COLS, CAT_COLS).fit(X_fit)
names = prep.get_feature_names_out()
n_cont = len(CONT_COLS)
check = pd.DataFrame({
    jeu: {"moyenne": prep.transform(X)[:, :n_cont].mean(), "écart-type": prep.transform(X)[:, :n_cont].std()}
    for jeu, X in [("Entraînement (X_fit)", X_fit), ("Validation", X_val), ("Test", X_test)]
}).T
print(f"Dimension après prétraitement : {len(names)} colonnes "
      f"({len(CONT_COLS)} continues, {len(DISC_COLS)} discrètes, {len(names) - len(CONT_COLS) - len(DISC_COLS)} one-hot)")
print("Moyenne et écart-type des variables continues transformées :")
check.style.format("{:.3f}")

## 2.4 Baseline

Le sujet demande de **construire une baseline avant les modèles plus complexes**. Une baseline est un modèle volontairement simple qui fixe le niveau minimal à dépasser : si un modèle avancé ne fait pas mieux, il n'apporte rien et sa complexité n'est pas justifiée.

**Baseline retenue : écart moyen en z-score au trafic normal.**

- *Principe* : après le prétraitement, chaque variable est exprimée en nombre d'écarts-types par rapport au trafic normal d'entraînement. Le score d'un flux est la **moyenne des écarts absolus** sur toutes ses variables. Plus il est élevé, plus le flux s'écarte du normal.
- *Pourquoi celle-ci* : aucun hyperparamètre, apprentissage limité à une moyenne et un écart-type, résultat explicable en une phrase à un analyste. Elle respecte la même contrainte que les détecteurs : elle n'apprend **que** sur du trafic normal.
- *Ce qu'elle ne sait pas faire* : elle traite chaque variable séparément et ignore les combinaisons (par exemple « beaucoup d'octets **sur un service** qui en échange habituellement peu »). Si les détecteurs des phases suivantes font mieux, c'est qu'ils exploitent ces combinaisons.
- *Alternative écartée* : une baseline « aucune alerte » est triviale (rappel nul) et n'apprend rien sur les données.

**Convention de score** : comme pour les détecteurs de scikit-learn, la méthode `score_samples` renvoie un score élevé pour un flux normal. On utilise donc `anomaly_score = −score_samples` : **plus il est grand, plus le flux est anormal**.

Cette cellule définit aussi les **outils communs** qui serviront à toutes les expériences : la règle de seuil et le calcul des métriques.

In [ ]:
class ZScoreBaseline(BaseEstimator):
    """Baseline : écart absolu moyen (en écarts-types) au trafic normal d'entraînement."""
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.mu_, self.sd_ = X.mean(axis=0), X.std(axis=0)
        self.sd_[self.sd_ == 0] = 1.0   # évite une division par zéro pour une variable constante
        return self

    def score_samples(self, X):
        # Convention scikit-learn : score élevé = normal. On renvoie donc l'opposé de l'écart moyen.
        X = np.asarray(X, dtype=float)
        return -np.abs((X - self.mu_) / self.sd_).mean(axis=1)


def anomaly_score(pipe, X):
    """Plus le score est élevé, plus le flux est anormal."""
    return -pipe.score_samples(X)


def threshold_from_normals(scores_normal, target_fpr=TARGET_FPR):
    """Seuil tel qu'une part target_fpr des flux normaux le dépasse.
    Exemple : target_fpr = 5 % -> 95e percentile des scores normaux de validation.
    Aucune attaque n'est utilisée (cohérent avec le scénario étudié)."""
    return np.quantile(scores_normal, 1 - target_fpr)


def detection_metrics(y, scores, thr):
    pred = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "Précision": precision_score(y, pred, zero_division=0),
        "Rappel": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "Macro-F1": f1_score(y, pred, average="macro", zero_division=0),
        "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
        "ROC-AUC": roc_auc_score(y, scores),
        "PR-AUC": average_precision_score(y, scores),
        "FP": int(fp), "FN": int(fn),
    }


PCT_COLS = ["Précision", "Rappel", "F1", "Macro-F1", "FPR", "ROC-AUC", "PR-AUC"]

def fmt(df):
    cols = [c for c in PCT_COLS if c in df.columns]
    return df.style.format({c: "{:.1%}" for c in cols}, precision=2)

## 2.5 Évaluation de la baseline

**Protocole** (identique à celui qu'utiliseront les détecteurs) :
1. le pipeline (prétraitement + baseline) est entraîné sur `X_fit`, uniquement des flux normaux ;
2. le seuil est fixé au 95e percentile des scores des flux **normaux de validation** (`TARGET_FPR` = 5 %) ;
3. on mesure les performances sur la validation, puis sur le test officiel, **sans rien modifier** entre les deux.

Le test sert ici de **point de référence** : ces chiffres seront la première ligne des tableaux de comparaison de l'expérience 1.

**À quoi s'attendre** :
- le FPR de validation vaut 5 % par construction (c'est ainsi qu'on a fixé le seuil) ;
- le FPR de test peut s'en écarter un peu : cet écart mesure la stabilité du seuil d'un jeu de données à l'autre ;
- une PR-AUC nettement supérieure à la proportion d'attaques signifie que la baseline fait mieux que le hasard.

In [ ]:
baseline = Pipeline([
    ("prep", build_preprocessor(CONT_COLS, DISC_COLS, CAT_COLS)),
    ("model", ZScoreBaseline()),
])
t0 = time.perf_counter()
baseline.fit(X_fit)
t_fit = time.perf_counter() - t0

s_val_bl, s_test_bl = anomaly_score(baseline, X_val), anomaly_score(baseline, X_test)
thr_bl = threshold_from_normals(s_val_bl[val_normal])

bl_results = pd.DataFrame({
    "Validation": detection_metrics(y_val, s_val_bl, thr_bl),
    "Test officiel": detection_metrics(y_test, s_test_bl, thr_bl),
}).T
save_table(bl_results, "phase2_baseline")
print(f"Baseline entraînée en {t_fit:.2f} s sur {len(X_fit)} flux normaux | seuil = {thr_bl:.3f}")
print(f"Proportion d'attaques (PR-AUC d'un modèle aléatoire) : validation {y_val.mean():.1%}, test {y_test.mean():.1%}")
fmt(bl_results)

### Distribution des scores et compromis détection / fausses alertes

- **Histogramme** : scores des flux normaux et des attaques du test, avec le seuil. Plus les deux distributions sont séparées, meilleure est la baseline ; leur chevauchement correspond aux attaques indiscernables du normal pour elle.
- **Rappel selon le budget de FPR** : seuils fixés sur la validation à 1, 2, 5 et 10 % de FPR. C'est la référence à laquelle les détecteurs seront comparés à budget égal.
- **Courbes ROC et précision-rappel** : vue complète, tous seuils confondus.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
lo, hi = np.quantile(s_test_bl, [0.001, 0.995])
bins = np.linspace(lo, hi, 80)
sc = np.clip(s_test_bl, lo, hi)
axes[0].hist(sc[y_test == 0], bins=bins, alpha=0.6, density=True, label="Normal")
axes[0].hist(sc[y_test == 1], bins=bins, alpha=0.6, density=True, label="Attaque")
axes[0].axvline(np.clip(thr_bl, lo, hi), color="black", ls="--", label=f"Seuil (FPR visé {TARGET_FPR:.0%})")
axes[0].set_title("Scores de la baseline (test)"); axes[0].set_xlabel("Score d'anomalie (écrêté pour l'affichage)")
axes[0].legend()
RocCurveDisplay.from_predictions(y_test, s_test_bl, name="Baseline z-score", ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", lw=1); axes[1].set_title("Courbe ROC (test)")
PrecisionRecallDisplay.from_predictions(y_test, s_test_bl, name="Baseline z-score", ax=axes[2])
axes[2].axhline(y_test.mean(), color="grey", ls=":", label="Hasard"); axes[2].legend()
axes[2].set_title("Courbe précision-rappel (test)")
plt.tight_layout(); save_fig(fig, "phase2_baseline_scores"); plt.show()

rows = []
for f in FPR_LEVELS:
    m = detection_metrics(y_test, s_test_bl, threshold_from_normals(s_val_bl[val_normal], f))
    rows.append({"FPR visé (validation)": f"{f:.0%}", "FPR obtenu (test)": m["FPR"],
                 "Rappel (test)": m["Rappel"], "Précision (test)": m["Précision"]})
bl_tradeoff = pd.DataFrame(rows).set_index("FPR visé (validation)")
save_table(bl_tradeoff, "phase2_baseline_compromis")
bl_tradeoff.style.format("{:.1%}")

### Détection par famille d'attaque (validation)

Quelles familles la baseline repère-t-elle facilement, lesquelles lui échappent ? Calcul sur la **validation**, pour ne pas multiplier les usages du test. La ligne *Normal* correspond au FPR.

Ce tableau donne une première idée des attaques qui « ressemblent au trafic normal » (question Q2 du sujet), à confirmer avec les détecteurs.

In [ ]:
pred_val = s_val_bl >= thr_bl
fam_bl = (pd.Series(pred_val, index=val_df.index).groupby(val_df["attack_cat"])
          .agg(taux_de_détection="mean", flux="count")
          .sort_values("taux_de_détection", ascending=False))
save_table(fam_bl, "phase2_baseline_par_famille")
fam_bl.style.format({"taux_de_détection": "{:.1%}"})

## 2.6 Bilan des phases 1 et 2

### Ce qui est prêt pour les expériences

| Élément | Où | Réutilisé par |
|---|---|---|
| Train, validation et test définis, contrôles de non-fuite | 2.2 | Toutes les expériences |
| Échantillon d'entraînement normal `X_fit` (`N_FIT` flux, sans doublon) | 2.2 | Expérience 1 ; base de l'expérience 2 |
| Pool d'attaques d'entraînement | 2.2 | Contamination (Exp. 2), classifieur supervisé (Exp. 3) |
| Pipeline de prétraitement sans fuite | 2.3 | Tous les modèles |
| Règle de seuil (`threshold_from_normals`) et métriques (`detection_metrics`) | 2.4 | Tous les modèles |
| Résultats de la baseline | 2.5 | Point de comparaison de l'expérience 1 |

### ✍️ Observations à rédiger avec vos chiffres

**Observation expérimentale** — Rappel, précision, FPR et PR-AUC de la baseline sur la validation et le test ; écart entre FPR visé et FPR obtenu sur le test ; familles les mieux et les moins bien détectées.

**Interprétation ML** — La baseline sépare-t-elle normal et attaque mieux que le hasard (PR-AUC comparée à la proportion d'attaques) ? Le chevauchement des scores indique la part d'attaques qui ne s'écartent pas du normal variable par variable.

**Interprétation cybersécurité** — Combien de fausses alertes produirait ce seuil sur le trafic réel d'une organisation, et pour quelle part d'attaques détectées ? Ce compromis est-il exploitable par un SOC ?

**Limites** — Baseline volontairement simple (variables traitées séparément) ; un seul échantillon d'entraînement ; seuil dépendant de la validation ; dataset simulé.

### Suite : phases 3 à 5
Les trois détecteurs (Isolation Forest, LOF, One-Class SVM) seront entraînés sur **le même `X_fit`**, avec **le même pipeline**, **la même règle de seuil** et **les mêmes métriques**, puis comparés à la baseline.